# Silver — POS sales

Bronze `sales` → `silver.sales` (SCD type 1).

Two awkward things about this feed:

1. **There is no transaction id.** The store export has no key at all, so we build
   one: a SHA-256 of the business columns. It is deterministic, so the same receipt
   always gets the same `sale_id` and the eight exact-duplicate rows in the export
   collapse into one.
2. **The product JSON does not parse.** About 8% of product names contain an
   unescaped `"`, so `from_json` returns NULL for those rows. We pull the fields out
   with regular expressions instead — uglier, but it loses no rows.

SCD1 because a receipt is a fact that never changes; there is no history to keep.

In [ ]:
import sys
from datetime import date
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "common_utils").is_dir():
        sys.path.insert(0, str(candidate))
        break

from pyspark.sql import functions as F

from common_utils.logger import get_logger, log_info
from common_utils.observability import ensure_ops_schema, new_run_id, track
from common_utils.scd import business_columns, deduplicate, row_hash, scd1_merge
from common_utils.settings import parse_run_date
from common_utils.transforms import add_derived, cast_columns, drop_columns, normalise_nulls, rename_columns, trim_columns
from common_utils.writers import cluster_by, create_namespace, qualified, set_table_properties

In [ ]:
dbutils.widgets.text("catalog", "retaildataplatform")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text("silver_schema", "silver")
dbutils.widgets.text("run_date", date.today().isoformat())

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
run_date = parse_run_date(dbutils.widgets.get("run_date"))
run_id = new_run_id()
logger = get_logger("silver")

create_namespace(spark, catalog, silver_schema, comment="Silver: cleaned, typed and de-duplicated entities with change history")
ensure_ops_schema(spark, catalog)

In [ ]:
bronze = spark.table(f"{catalog}.{bronze_schema}.sales").filter(F.col("_load_date") == F.lit(run_date).cast("date"))
print("bronze rows for", run_date, ":", bronze.count())

## 1. Rename and type
`product_name` in the export disagrees with the name inside the JSON payload for
~70% of rows, and the payload is authoritative — so the export's column is kept
under an honest name, `listed_product_name`.

In [ ]:
cleaned = normalise_nulls(bronze)
cleaned = trim_columns(cleaned, ["customer_name", "product_name", "product_category"])
cleaned = rename_columns(
    cleaned,
    {
        "product_name": "listed_product_name",
        "product_category": "brand",
        "total_price": "total_amount",
        "product": "product_json",
    },
)
cleaned = cast_columns(cleaned, {"customer_id": "bigint", "order_date": "date", "total_amount": "decimal(14,2)"})

## 2. Pull the product fields out of the JSON text
`regexp_extract(text, pattern, 1)` returns capture group 1, or `''` if it does not
match — which is why the quality check on `product_id` matters.

In [ ]:
parsed = add_derived(
    cleaned,
    {
        "product_id": r"""regexp_extract(product_json, '"id":"([^"]+)"', 1)""",
        "product_name": r"""regexp_extract(product_json, '"name":"(.*)","price"', 1)""",
        "unit_price": r"""CAST(regexp_extract(product_json, '"price":([0-9.]+)', 1) AS DECIMAL(12,2))""",
        "quantity": r"""CAST(regexp_extract(product_json, '"qty":([0-9]+)', 1) AS INT)""",
        "currency": r"""regexp_extract(product_json, '"curr":"([^"]+)"', 1)""",
    },
)

## 3. Build the key
No natural key upstream, so hash the business columns. Identical rows hash
identically and collapse — which is exactly what we want for a duplicated export.

In [ ]:
keyed = add_derived(
    parsed,
    {
        "sale_id": "sha2(concat_ws('|', CAST(customer_id AS STRING), CAST(order_date AS STRING), product_json, listed_product_name, CAST(total_amount AS STRING)), 256)"
    },
)
keyed = drop_columns(keyed, ["product_json", "_ingested_at", "_source_file"])

In [ ]:
target = qualified(catalog, silver_schema, "sales")

with track(spark, catalog, run_id, run_date, task="sales_silver", layer="silver", entity="sales") as stats:
    hashed = row_hash(keyed, business_columns(keyed))
    deduped = deduplicate(hashed, keys=["sale_id"])
    prepared = deduped.withColumn("_updated_at", F.current_timestamp())

    stats.rows_read = prepared.count()
    scd1_merge(spark, prepared, target, keys=["sale_id"])
    stats.rows_written = stats.rows_read

    set_table_properties(spark, catalog, silver_schema, "sales")
    cluster_by(spark, catalog, silver_schema, "sales", ["order_date", "customer_id"])
    log_info(logger, "silver sales merged", rows=stats.rows_read)

In [ ]:
display(
    spark.sql(
        f"""
        SELECT count(*) AS rows,
               count(DISTINCT sale_id) AS distinct_sales,
               sum(CASE WHEN product_id = '' OR product_id IS NULL THEN 1 ELSE 0 END) AS unparsed_products,
               sum(CASE WHEN total_amount <> unit_price * quantity THEN 1 ELSE 0 END) AS amount_mismatches
        FROM {catalog}.{silver_schema}.sales
        """
    )
)